# 第 18 节：GAE (Generalized Advantage Estimation)

---

## 📍 本节位置

```
Actor-Critic (14) → 重要性采样 (16) → **GAE (18)** → PPO 理论 (19) → PPO 实战 (20)
                                           ↑
                                       你在这里
```

GAE 是 Schulman et al. (2016) 提出的优势函数估计方法，它通过一个参数 $\lambda$ 平滑地在偏差和方差之间进行插值，是现代 Actor-Critic 方法（如 PPO、A2C）的标准组件。


## 🎯 学习目标

1. 理解 $\lambda$-return 的定义及其与 TD(0) 和 MC 的关系
2. 掌握 GAE 的推导过程（从 $\lambda$-return 到 TD 残差的和）
3. 区分 $\gamma$（折扣因子）和 $\lambda$（偏差-方差权衡参数）
4. 实现 GAE 的递推计算
5. 通过实验观察不同 $\lambda$ 值对优势估计的影响
6. 学会在实际项目中调优 $\lambda$ 参数


## 动机：偏差与方差的权衡

### 1-step TD 估计（高偏差，低方差）

$$\hat{A}_t^{(1)} = \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

- 只使用一步实际奖励，其余依赖 $V$ 的估计（bootstrap）
- **高偏差**：$V$ 的估计误差直接影响优势
- **低方差**：只用一步随机性

### Monte Carlo 估计（低偏差，高方差）

$$\hat{A}_t^{(\infty)} = \sum_{l=0}^{\infty} \gamma^l r_{t+l} - V(s_t)$$

- 使用完整的实际回报
- **低偏差**：不需要 bootstrap
- **高方差**：累积了多步的随机性

### n-step 估计

$$\hat{A}_t^{(n)} = \sum_{l=0}^{n-1} \gamma^l r_{t+l} + \gamma^n V(s_{t+n}) - V(s_t)$$

- 随着 $n$ 增大：偏差降低，方差增大
- 问题：选择哪个 $n$ 最合适？

### 我们的目标

> 不要选择一个固定的 $n$，而是**对所有 $n$ 进行加权平均**——这就是 $\lambda$-return 的思想。


## $\lambda$-return：对所有 n-step 回报的加权平均

### 定义

$$G_t^{\lambda} = (1 - \lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)}$$

其中 $G_t^{(n)}$ 是 n-step 回报：

$$G_t^{(n)} = \sum_{l=0}^{n-1} \gamma^l r_{t+l} + \gamma^n V(s_{t+n})$$

### 权重分布

权重序列 $(1-\lambda), (1-\lambda)\lambda, (1-\lambda)\lambda^2, \ldots$ 构成一个几何分布，总和为 1：

$$\sum_{n=1}^{\infty} (1-\lambda)\lambda^{n-1} = 1$$

### 极端情况

| $\lambda$ | 等价于 | 偏差 | 方差 |
|:---------:|:------:|:----:|:----:|
| $\lambda = 0$ | TD(0) / 1-step return | 最高 | 最低 |
| $\lambda = 1$ | Monte Carlo return | 最低 | 最高 |

### $\lambda$ 的含义

> $\lambda$ 控制了我们**对 bootstrap 的信任程度**：
> - $\lambda \to 0$：更信任价值网络的估计（强 bootstrap）
> - $\lambda \to 1$：更信任实际奖励信号（弱 bootstrap）


## GAE 优势估计：从 $\lambda$-return 到 TD 残差之和

### 从 $\lambda$-return 到优势

GAE 的优势定义为：

$$A_t^{\text{GAE}(\gamma, \lambda)} = G_t^{\lambda} - V(s_t)$$

### 推导过程

从定义出发，将 $G_t^{\lambda}$ 展开：

$$
\begin{aligned}
A_t^{\text{GAE}}
&= G_t^{\lambda} - V(s_t) \\
&= (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} G_t^{(n)} - V(s_t) \\
&= (1-\lambda) \sum_{n=1}^{\infty} \lambda^{n-1} \left(G_t^{(n)} - V(s_t)\right)
\end{aligned}
$$

代入 $G_t^{(n)}$：

$$G_t^{(n)} - V(s_t) = \sum_{l=0}^{n-1} \gamma^l r_{t+l} + \gamma^n V(s_{t+n}) - V(s_t)$$

### 最终形式

经过代数简化，得到 GAE 的美妙形式——**TD 残差的指数加权和**：

$$A_t^{\text{GAE}(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \, \delta_{t+l}$$

其中 TD 残差为：

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

### 递推形式

GAE 可以高效递推计算：

$$A_t = \delta_t + \gamma \lambda \, A_{t+1}$$

这使我们可以在 $O(T)$ 时间内计算整条轨迹的所有优势值。


## 理解 $\gamma$ 和 $\lambda$ 的独立作用

### $\gamma$（折扣因子）：时间尺度

$$\gamma \in [0, 1]$$

- 控制**多远**的未来奖励被考虑
- $\gamma \to 0$：只看即时奖励（短视）
- $\gamma \to 1$：考虑长远奖励（远视）
- **影响的是价值函数的定义**本身

### $\lambda$（GAE 参数）：偏差-方差权衡

$$\lambda \in [0, 1]$$

- 控制**如何**估计优势
- $\lambda \to 0$：强 bootstrap（高偏差，低方差）→ TD(0)
- $\lambda \to 1$：少 bootstrap（低偏差，高方差）→ MC
- **不影响价值函数的定义**，只影响估计方法

### 两者的关系

| | $\gamma$ 小 | $\gamma$ 大 |
|:----:|:-----------:|:----------:|
| $\lambda$ 小 | 短视 + 强 bootstrap | 远视 + 强 bootstrap |
| $\lambda$ 大 | 短视 + 弱 bootstrap | 远视 + 弱 bootstrap |

### 乘积 $\gamma \lambda$ 的实际含义

$$(\gamma\lambda)^l \, \delta_{t+l}$$

- $\gamma \lambda$ 一起控制了 TD 残差的衰减速度
- $\gamma \lambda$ 越小：高阶残差衰减越快 → 更依赖早期 bootstrap
- $\gamma \lambda$ 越大：高阶残差衰减越慢 → 更依赖长期实际奖励

### 典型配置

| 算法 | $\gamma$ | $\lambda$ |
|:----:|:---------:|:---------:|
| PPO | 0.99 | 0.95 |
| A2C | 0.99 | 1.0 (MC) |
| IMPALA | 0.99 | 0.95 |
| GA3C | 0.99 | 1.0 |


In [ ]:
import os
FAST_MODE = os.getenv('RL_COURSE_FAST_MODE', '0') == '1'
# ============================================================
# 从零实现 GAE：逐步骤计算
# ============================================================
# 本实现展示 GAE 的两种计算方式：
#   1. 原始定义：从 λ-return 推导
#   2. 递推形式：O(T) 高效计算

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def compute_gae_definition(rewards, values, done, gamma=0.99, lam=0.95):
    """
    根据原始定义计算 GAE（用于教学演示）
    A_t = Σ_{l=0}^{T-t-1} (γλ)^l · δ_{t+l}

    Args:
        rewards: (T,) 奖励序列 r_0,...,r_{T-1}
        values:   (T+1,) 价值序列 V(s_0),...,V(s_T)
        done:     (T,) done 标志
    Returns:
        advantages: (T,) 优势估计
    """
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_value = values[t+1] * (1 - done[t])  # 终止状态后 V=0
        deltas[t] = rewards[t] + gamma * next_value - values[t]

    advantages = np.zeros(T)
    for t in range(T):
        # 对每个时间步，累加衰减的 δ
        gael = 0.0
        power = 1.0
        for l in range(T - t):
            gael += power * deltas[t + l]
            power *= gamma * lam
        advantages[t] = gael
    return advantages, deltas

def compute_gae_recursive(rewards, values, done, gamma=0.99, lam=0.95):
    """
    递推计算 GAE：O(T) 高效版本
    A_t = δ_t + γλ · A_{t+1} · (1 - done_t)

    Args:
        rewards: (T,) 奖励序列
        values:   (T+1,) 价值序列
        done:     (T,) done 标志
    Returns:
        advantages: (T,) 优势估计
    """
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_value = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_value - values[t]

    advantages = np.zeros(T)
    advantages[-1] = deltas[-1]
    for t in reversed(range(T - 1)):
        advantages[t] = deltas[t] + gamma * lam * advantages[t+1] * (1 - done[t])
    return advantages, deltas

# 测试：简单轨迹
T = 6
rewards = np.array([1.0, 2.0, -1.0, 3.0, 0.5, 1.0])
values  = np.array([0.5, 1.2, 0.8, 0.3, 1.5, 0.9, 0.4])
done    = np.array([0, 0, 0, 0, 0, 1])  # 最后一步 done

adv_def, deltas_def = compute_gae_definition(rewards, values, done)
adv_rec, deltas_rec = compute_gae_recursive(rewards, values, done)

print("=== GAE 计算结果对比 ===")
print(f"{'t':<4} {'r_t':<8} {'V_t':<8} {'δ_t':<8} {'A_t(def)':<12} {'A_t(rec)':<12}")
for t in range(T):
    print(f"{t:<4} {rewards[t]:<8.2f} {values[t]:<8.2f} {deltas_def[t]:<8.2f} "
          f"{adv_def[t]:<12.4f} {adv_rec[t]:<12.4f}")

print(f"\n两种方法一致: {np.allclose(adv_def, adv_rec)}")


In [ ]:
# ============================================================
# GAE 在不同 λ 下的对比：5 步 MDP 示例
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 构造轨迹数据
T = 10  # 轨迹长度
rewards = np.array([-0.5, 1.0, 1.5, -0.2, 2.0, -1.0, 3.0, 0.5, 0.0, 2.0])
values  = np.array([0.2, 0.8, 1.1, 0.6, 1.8, 0.3, 2.2, 1.5, 1.0, 1.8, 0.9])
done    = np.zeros(T)
done[-1] = 1.0

def compute_gae(rewards, values, done, gamma=0.99, lam=0.95):
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_value = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_value - values[t]
    advantages = np.zeros(T)
    for t in reversed(range(T)):
        if t == T - 1:
            advantages[t] = deltas[t]
        else:
            advantages[t] = deltas[t] + gamma * lam * advantages[t+1] * (1 - done[t])
    return advantages

# 比较不同 λ
lambdas = [0.0, 0.5, 0.8, 0.95, 1.0]
gamma = 0.99

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

all_advs = {}
for lam in lambdas:
    adv = compute_gae(rewards, values, done, gamma=gamma, lam=lam)
    all_advs[lam] = adv
    axes[0].plot(range(T), adv, marker='o', label=f'λ={lam}', linewidth=1.5)

axes[0].set_xlabel('时间步 t'); axes[0].set_ylabel('A_t (GAE)')
axes[0].set_title(f'不同 λ 下的 GAE 优势估计 (γ={gamma})')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 方差 vs 偏差的直观展示
for lam in lambdas:
    axes[1].bar(lam, np.var(all_advs[lam]), width=0.1, alpha=0.7,
                label=f'λ={lam}: var={np.var(all_advs[lam]):.3f}')
axes[1].set_xlabel('λ'); axes[1].set_ylabel('优势估计的方差')
axes[1].set_title('λ 增大 → 方差增大')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/18_gae_lambda_comparison.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_lambda_comparison.png")

# 打印对比表
print("=== 不同 λ 的 GAE 对比 ===")
print(f"{'t':<4}", end='')
for lam in lambdas:
    print(f"{f'λ={lam}':<10}", end='')
print()
for t in range(T):
    print(f"{t:<4}", end='')
    for lam in lambdas:
        print(f"{all_advs[lam][t]:<10.3f}", end='')
    print()
print(f"\n{'方差':<6}", end='')
for lam in lambdas:
    print(f"{np.var(all_advs[lam]):<10.4f}", end='')
print()


In [ ]:
# ============================================================
# 可视化不同 λ 的 GAE 热力图
# ============================================================

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 生成更长的随机轨迹
T = 20
true_advantages = np.sin(np.linspace(0, 4*np.pi, T)) * 2  # 模拟真实的优势变化

# 添加随机奖励和价值的生成
rewards = np.random.randn(T) * 0.5 + 0.2
values_true = 0.5 * np.sin(np.linspace(0, 4*np.pi, T+1)) + 1
values = values_true + np.random.randn(T+1) * 0.3  # 带噪声的价值估计
done = np.zeros(T)
done[-1] = 1

def compute_gae(rewards, values, done, gamma=0.99, lam=0.95):
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_v = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_v - values[t]
    advantages = np.zeros(T)
    for t in reversed(range(T)):
        if t == T - 1:
            advantages[t] = deltas[t]
        else:
            advantages[t] = deltas[t] + gamma * lam * advantages[t+1] * (1 - done[t])
    return advantages

lambdas = [0.0, 0.25, 0.5, 0.75, 0.95, 1.0]
gamma = 0.99

fig, ax = plt.subplots(figsize=(12, 6))

adv_matrix = np.zeros((len(lambdas), T))
for i, lam in enumerate(lambdas):
    adv_matrix[i] = compute_gae(rewards, values, done, gamma, lam)

im = ax.imshow(adv_matrix, aspect='auto', cmap='RdBu_r', interpolation='nearest')
ax.set_yticks(range(len(lambdas)))
ax.set_yticklabels([f'λ={lam}' for lam in lambdas])
ax.set_xlabel('时间步 t')
ax.set_ylabel('λ 值')
ax.set_title('GAE 优势估计热力图：不同 λ 的比较')

# 在每个格子中显示数值
for i in range(len(lambdas)):
    for j in range(T):
        val = adv_matrix[i, j]
        color = 'white' if abs(val) > 0.5 else 'black'
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='A_t')
plt.tight_layout()
plt.savefig('outputs/figures/18_gae_heatmap.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_heatmap.png")

# 显示 λ=0 和 λ=1 的差异
adv_td = adv_matrix[0]  # λ=0
adv_mc = adv_matrix[-1]  # λ=1

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(adv_td, 'b-o', label=f'TD(0) / GAE(λ=0)', alpha=0.7)
ax.plot(adv_mc, 'r-s', label=f'MC / GAE(λ=1)', alpha=0.7)
ax.fill_between(range(T), adv_td, adv_mc, alpha=0.15, color='purple',
                label='TD~MC 差异区域')
ax.set_xlabel('时间步 t'); ax.set_ylabel('A_t')
ax.set_title('λ=0 (TD) vs λ=1 (MC) 的优势估计比较')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/figures/18_gae_td_vs_mc.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_td_vs_mc.png")


## $\lambda$ 对训练的影响

### 低 $\lambda$（接近 0）：高偏差，低方差

**行为特征**：
- 优势估计主要依赖当前 TD 残差 $\delta_t$
- 价值网络的偏差直接影响策略梯度
- 每个时间步的优势独立，受噪声影响小

**训练表现**：
- 学习曲线更平滑（低方差）
- 可能收敛到次优策略（高偏差）
- 对价值网络的质量高度敏感

**适合场景**：
- 价值网络已经很准确时
- 需要稳定训练时
- 奖励信号很稀疏时（避免引入过多噪声）

### 高 $\lambda$（接近 1）：低偏差，高方差

**行为特征**：
- 优势估计接近 MC 回报
- 实际奖励占主导，较少依赖价值网络
- 多步累积的随机性导致高方差

**训练表现**：
- 学习曲线波动较大（高方差）
- 更容易找到最优策略（低偏差）
- 需要更多样本来平均方差

**适合场景**：
- 奖励信号提供清晰的指导
- 价值网络不够准确时
- 有足够样本时

### 实际经验

> 在大多数连续控制任务中，$\lambda = 0.95$ 是默认选择。它提供了适度的偏差校正，同时保持了可接受的方差水平。在视觉观察的任务（如 Atari）中，$\lambda$ 常设置在 0.95-0.99 之间。


In [ ]:
# ============================================================
# 在 CartPole 上使用不同 λ 训练 PPO
# ============================================================
# 注意：这是一个轻量级演示，仅跑少量步数以展示 λ 的影响
# 完整的训练需要更长的步数来体现差异

import sys; sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath("__file__")))), os
import numpy as np
import torch
import gymnasium as gym
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
from rl_course.agents.ppo import PPOAgent
from rl_course.utils.seeding import set_seed

# 创建环境
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(f"CartPole: state_dim={state_dim}, n_actions={n_actions}")

# 要比较的 λ 值
lambda_values = [0.0, 0.5, 0.95, 1.0]
n_updates = 5 if FAST_MODE else 20  # 每个 agent 的训练步数（缩短以加快演示）
n_steps = 128 if FAST_MODE else 512   # 每次收集的步数

all_rewards = {}

for lam in lambda_values:
    set_seed(42)
    agent = PPOAgent(
        state_dim=state_dim,
        n_actions=n_actions,
        gamma=0.99,
        gae_lambda=lam,
        clip_epsilon=0.2,
        lr=3e-4,
        n_steps=n_steps,
        batch_size=64,
        n_epochs=5,
    )

    episode_rewards = []
    obs, _ = env.reset(seed=42)
    ep_return = 0

    for update in range(n_updates):
        for step in range(agent.n_steps):
            action = agent.act(obs)
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 计算 next_value（必须在 env.reset() 之前！）
            with torch.no_grad():
                if terminated:
                    next_value = 0.0
                else:
                    next_obs_t = torch.as_tensor(
                        next_obs, dtype=torch.float32, device=agent.device
                    ).unsqueeze(0)
                    next_value = agent.network.get_value(next_obs_t).item()

            agent.store(reward, done, terminated, next_value)

            ep_return += reward
            if done:
                episode_rewards.append(ep_return)
                ep_return = 0
                obs, _ = env.reset()
            else:
                obs = next_obs

        # 更新
        metrics = agent.update()

        if (update + 1) % 5 == 0:
            print(f"  λ={lam}, 更新 {update+1}/{n_updates}, "
                  f"clip={metrics['pre_step_clip_fraction']:.3f}, "
                  f"KL={metrics['pre_step_approx_kl']:.4f}")

    all_rewards[lam] = episode_rewards
    print(f"λ={lam} 完成, 共 {len(episode_rewards)} 个 episode")

env.close()

# 绘制结果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lam, rewards in all_rewards.items():
    if len(rewards) > 0:
        axes[0].plot(rewards, alpha=0.5, label=f'λ={lam}', linewidth=0.8)

axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Return')
axes[0].set_title('不同 λ 的 PPO 训练曲线 (CartPole)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# 平均回报对比
mean_rewards = {lam: np.mean(rewards[-10:]) if len(rewards) >= 10
                else np.mean(rewards) for lam, rewards in all_rewards.items()}
bars = axes[1].bar(list(mean_rewards.keys()), list(mean_rewards.values()),
                   width=0.15, alpha=0.7)
axes[1].set_xlabel('λ'); axes[1].set_ylabel('平均回报 (最后 10 ep)')
axes[1].set_title('λ 对最终性能的影响')
for bar, (lam, val) in zip(bars, mean_rewards.items()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}', ha='center', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('outputs/figures/18_ppo_lambda_comparison.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_ppo_lambda_comparison.png")


In [ ]:
# ============================================================
# λ 值对比的详细可视化
# ============================================================
# 对每个 λ，显示 GAE 公式的逐步分解

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

gamma = 0.99

# 模拟一个场景：突然的大奖励
T = 15
rewards = np.zeros(T)
rewards[7] = 5.0  # 在第 7 步有一个大的正奖励
values = np.ones(T + 1) * 0.5
done = np.zeros(T)
done[-1] = 1

def compute_gae_full(rewards, values, done, gamma, lam):
    """返回 GAE 和各个组成部分"""
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_v = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_v - values[t]

    advantages = np.zeros(T)
    components = np.zeros((T, T))  # components[t, l] = (γλ)^l · δ_{t+l}
    for t in range(T):
        power = 1.0
        for l in range(T - t):
            components[t, l] = power * deltas[t + l]
            power *= gamma * lam
        advantages[t] = np.sum(components[t, :T-t])
    return advantages, deltas, components

lambdas_to_plot = [0.0, 0.5, 0.95, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, lam in enumerate(lambdas_to_plot):
    adv, deltas, comp = compute_gae_full(rewards, values, done, gamma, lam)

    ax = axes[idx]
    # 绘制每个时间步的 A_t 分解
    for t in range(T):
        contributions = comp[t, :T-t]
        ax.bar(range(t, t + len(contributions)), contributions,
               alpha=0.6, label=f'A_{t}' if t == 0 else '')

    ax.plot(range(T), adv, 'ro-', linewidth=2, label='GAE A_t', zorder=5)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.5)
    ax.set_xlabel('时间步'); ax.set_ylabel('优势值')
    ax.set_title(f'GAE 分解 (λ={lam}, γ={gamma})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/18_gae_decomposition.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_decomposition.png")

print("=== 不同 λ 下的 GAE 总结 ===")
print(f"{'λ':<6} {'A_0':<10} {'A_3':<10} {'A_7':<10} {'A_14':<10} {'平均 |A|':<10}")
for lam in lambdas_to_plot:
    adv, _, _ = compute_gae_full(rewards, values, done, gamma, lam)
    print(f"{lam:<6.2f} {adv[0]:<10.3f} {adv[3]:<10.3f} {adv[7]:<10.3f} "
          f"{adv[14]:<10.3f} {np.mean(np.abs(adv)):<10.3f}")


## GAE 在实际中的使用

### 典型 $\lambda$ 值

| 任务类型 | 推荐 $\lambda$ | 理由 |
|:--------:|:--------------:|:------|
| 简单控制（CartPole, LunarLander） | 0.90 - 0.95 | 中等偏上的 bootstrap |
| 连续控制（MuJoCo, PyBullet） | 0.95 | 默认值，兼顾稳定和准确 |
| Atari 游戏 | 0.95 - 0.99 | 高维观察，需要低偏差 |
| 机器人控制 | 0.90 - 0.95 | 奖励信号含噪声，适当 bootstrap |
| 稀疏奖励任务 | 0.80 - 0.90 | 更多 bootstrap 帮助传播奖励 |

### 调优建议

1. **从 $\lambda = 0.95$ 开始**，这是大多数实现中的默认值
2. 如果训练不稳定，**降低 $\lambda$**（更多 bootstrap）提高稳定性
3. 如果收敛缓慢或性能不佳，**提高 $\lambda$** 降低偏差
4. 与 $\gamma$ 一起调优：通常先确定 $\gamma$，再调 $\lambda$

### 实际实现

```python
# GAE 递推实现的伪代码
def compute_gae(rewards, values, dones, gamma, lam):
    T = len(rewards)
    advantages = np.zeros(T)
    gae = 0
    for t in reversed(range(T)):
        delta = rewards[t] + gamma * values[t+1] * (1 - dones[t]) - values[t]
        gae = delta + gamma * lam * gae * (1 - dones[t])
        advantages[t] = gae
    returns = advantages + values[:T]
    return advantages, returns
```

### 与价值损失的关系

GAE 不仅用于策略梯度，还用于价值网络训练：

$$\mathcal{L}^{\text{VF}} = \mathbb{E}_t\left[\left(V_\theta(s_t) - (A_t^{\text{GAE}} + V(s_t))\right)^2\right]$$

其中 $G_t = A_t^{\text{GAE}} + V(s_t)$ 是价值网络的学习目标。


## 总结表：$\lambda$ 谱从 TD(0) 到 MC

### $\lambda$ 谱

```
TD(0)                  MC
λ = 0  ──── λ = 0.5 ──── λ = 0.95 ──── λ = 1.0
 │                        │               │
 │                        │               │
高偏差                   中间            低偏差
低方差                   平衡            高方差
强 bootstrap                             弱 bootstrap
```

### 各 $\lambda$ 详细对比

| 特征 | $\lambda = 0$ (TD(0)) | $\lambda = 0.5$ | $\lambda = 0.95$ (PPO 默认) | $\lambda = 1.0$ (MC) |
|:----:|:---------------------:|:----------------:|:---------------------------:|:---------------------:|
| 公式 | $\delta_t$ | $\sum_{l=0}^{\infty}0.5^l \gamma^l \delta_{t+l}$ | $\sum_{l=0}^{\infty}0.95^l \gamma^l \delta_{t+l}$ | $\sum_{l=0}^{\infty} \gamma^l \delta_{t+l}$ |
| 所需步数 | 1 | ~10（权重衰减快）| ~100 | 完整轨迹 |
| 偏差 | 高 | 中 | 低 | 无 |
| 方差 | 低 | 中 | 中高 | 高 |
| 对 V 的依赖 | 强 | 中 | 弱 | 无 |
| 计算成本 | $O(1)$ | $O(T)$ | $O(T)$ | $O(T)$ |
| 适用场景 | 价值网络准确时 | 一般场景 | 大多数 RL 任务 | 低偏差需求时 |

### 关键公式回顾

$$A_t^{\text{GAE}(\gamma, \lambda)} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \, \delta_{t+l}$$

$$\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

> **记住**：$\gamma$ 影响**回报的定义**（价值函数的目标），$\lambda$ 影响**优势的估计方式**（如何逼近这个目标）。


## 偏差-方差权衡的数学分析

### GAE 的偏差

GAE 的偏差来源于价值函数的 bootstrap。对于 $\hat{A}_t^{\text{GAE}}$：

$$\text{Bias}[\hat{A}_t^{\text{GAE}}] = \sum_{l=0}^{\infty} (\gamma\lambda)^l \, \mathbb{E}[\delta_{t+l} - \delta_{t+l}^*]$$

其中 $\delta_t^* = r_t + \gamma V^\pi(s_{t+1}) - V^\pi(s_t)$ 是使用真实价值函数时的 TD 残差。

当 $\lambda \to 1$ 时，偏差趋近于 0（MC 无偏估计）。
当 $\lambda \to 0$ 时，偏差等于 $\mathbb{E}[\delta_t - \delta_t^*]$（TD 偏差）。

### GAE 的方差

$$\text{Var}[\hat{A}_t^{\text{GAE}}] = \sum_{l=0}^{\infty} (\gamma\lambda)^{2l} \, \text{Var}[\delta_{t+l}] + 2\sum_{i<j} (\gamma\lambda)^{i+j} \, \text{Cov}[\delta_{t+i}, \delta_{t+j}]$$

当 $\lambda$ 增大时：
- 更多的 TD 残差被加和进来 → 方差增大
- 但 $\gamma\lambda$ 的幂次使高阶项衰减 → 方差可控

### 最优 $\lambda$

理论上存在一个最优 $\lambda$ 最小化 MSE：

$$\lambda^* = \arg\min_\lambda \; \mathbb{E}\left[\left(A_t^{\text{GAE}(\lambda)} - A_t^*\right)^2\right]$$

这个最优值取决于：
- 价值函数的准确度（越准确 → $\lambda$ 越小）
- 环境的随机性（随机性越大 → $\lambda$ 越小）
- 轨迹长度（越长 → $\lambda$ 需仔细调）


In [ ]:
# ============================================================
# 经验偏差-方差权衡：GAE 多次运行
# ============================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 构造一个随机环境模拟
def simulate_trajectory(T=20):
    '''生成一条随机轨迹和对应的价值估计'''
    true_returns = np.cumsum(np.random.randn(T) * 0.5)
    # 价值估计有误差
    values_true = true_returns + np.random.randn(T+1) * 0.1
    # 价值网络估计（有偏）
    values_est = values_true + np.random.randn(T+1) * 0.3
    rewards = np.diff(true_returns, prepend=0) + np.random.randn(T) * 0.2
    done = np.zeros(T)
    done[-1] = 1
    return rewards, values_est, values_true, done

def compute_gae(rewards, values, done, gamma=0.99, lam=0.95):
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_v = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_v - values[t]
    advantages = np.zeros(T)
    for t in reversed(range(T)):
        if t == T - 1:
            advantages[t] = deltas[t]
        else:
            advantages[t] = deltas[t] + gamma * lam * advantages[t+1] * (1 - done[t])
    return advantages

# 使用真实价值函数计算"真实"优势
def compute_true_advantage(rewards, values_true, done, gamma=0.99):
    '''使用真实价值计算 MC 优势'''
    T = len(rewards)
    advantages = np.zeros(T)
    G = 0
    for t in reversed(range(T)):
        G = rewards[t] + gamma * G * (1 - done[t])
        advantages[t] = G - values_true[t]
    return advantages

n_runs = 500
lambdas = np.linspace(0, 1, 11)
gamma = 0.99
T = 15

all_biases = {lam: [] for lam in lambdas}
all_vars = {lam: [] for lam in lambdas}

for _ in range(n_runs):
    rewards, values_est, values_true, done = simulate_trajectory(T)
    true_adv = compute_true_advantage(rewards, values_true, done, gamma)

    for lam in lambdas:
        est_adv = compute_gae(rewards, values_est, done, gamma, lam)
        # 计算偏差和方差（对所有时间步平均）
        bias = np.mean(est_adv - true_adv)
        var = np.var(est_adv)
        all_biases[lam].append(bias)
        all_vars[lam].append(var)

# 聚合
mean_bias = [np.mean(all_biases[lam]) for lam in lambdas]
mean_var = [np.mean(all_vars[lam]) for lam in lambdas]
mse = [b**2 + v for b, v in zip(mean_bias, mean_var)]

fig, ax1 = plt.subplots(figsize=(10, 5))

color1, color2, color3 = 'blue', 'red', 'green'
ax1.plot(lambdas, mean_bias, 'o-', color=color1, linewidth=2, label='Bias²')
ax1.plot(lambdas, mean_var, 's-', color=color2, linewidth=2, label='Variance')
ax1.plot(lambdas, mse, '^-', color=color3, linewidth=2, label='MSE')
ax1.set_xlabel('λ'); ax1.set_ylabel('误差')
ax1.set_title('GAE 的偏差-方差权衡：随 λ 的变化')
ax1.legend(); ax1.grid(True, alpha=0.3)

# 标注最佳 λ
best_idx = np.argmin(mse)
ax1.axvline(lambdas[best_idx], color='gray', linestyle='--', alpha=0.5)
ax1.annotate(f'最佳 λ={lambdas[best_idx]:.2f}',
            xy=(lambdas[best_idx], mse[best_idx]),
            xytext=(lambdas[best_idx]+0.1, mse[best_idx]+0.05),
            arrowprops=dict(arrowstyle='->'), fontsize=10)

plt.tight_layout()
plt.savefig('outputs/figures/18_gae_bias_variance.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_bias_variance.png")

print("=== 偏差-方差分析表 ===")
print(f"{'λ':<8} {'Bias²':<10} {'Variance':<12} {'MSE':<10}")
for lam, b, v, m in zip(lambdas, mean_bias, mean_var, mse):
    print(f"{lam:<8.2f} {b**2:<+10.4f} {v:<12.6f} {m:<10.6f}")
print(f"\n最佳 λ = {lambdas[best_idx]:.2f} (最小 MSE = {mse[best_idx]:.6f})")


## GAE 在连续控制和多步估计中的应用

### GAE 在连续动作空间

在连续控制中，GAE 的使用与离散控制完全相同——GAE 只需要奖励和价值函数，不需要动作信息：

$$A_t^{\text{GAE}} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \left(r_{t+l} + \gamma V(s_{t+l+1}) - V(s_{t+l})\right)$$

因此 GAE 可以无缝应用于 DDPG、SAC、TD3 等连续控制算法。

### GAE vs V-trace

| 特性 | GAE | V-trace (IMPALA) |
|:----:|:---:|:-----------------:|
| 设计目标 | 估计优势 | 估计价值函数 |
| 核心公式 | $\sum (\gamma\lambda)^l \delta_{t+l}$ | $\delta_t + \gamma \min(\bar{\rho}_t, 1) (v_{t+1} - V(s_{t+1}))$ |
| 重要性采样 | 无（on-policy） | 有（off-policy 截断）|
| 截断 | 无需 | $\bar{\rho}_t = \min(\rho_t, \bar{c})$ |
| 应用算法 | PPO, A2C | IMPALA |

### GAE 与 Retrace

Retrace($\lambda$) 是 GAE 的 off-policy 版本：

$$\Delta_t^{\text{Retrace}} = \sum_{l=0}^{\infty} (\gamma\lambda)^l \left(\prod_{j=1}^{l} c_{t+j}\right) \delta_{t+l}$$

其中 $c_t = \min\left(1, \frac{\pi(a_t \mid s_t)}{\mu(a_t \mid s_t)}\right)$ 是截断的重要性权重。

当 $\pi = \mu$（on-policy）时，$c_t = 1$，Retrace 退化为 GAE。


In [ ]:
# ============================================================
# GAE 计算效率分析
# ============================================================
import numpy as np
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 比较定义法和递推法的计算时间

def gae_definition(rewards, values, done, gamma=0.99, lam=0.95):
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_v = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_v - values[t]
    advantages = np.zeros(T)
    for t in range(T):
        gael = 0.0
        power = 1.0
        for l in range(T - t):
            gael += power * deltas[t + l]
            power *= gamma * lam
        advantages[t] = gael
    return advantages

def gae_recursive(rewards, values, done, gamma=0.99, lam=0.95):
    T = len(rewards)
    deltas = np.zeros(T)
    for t in range(T):
        next_v = values[t+1] * (1 - done[t])
        deltas[t] = rewards[t] + gamma * next_v - values[t]
    advantages = np.zeros(T)
    gae = 0
    for t in reversed(range(T)):
        gae = deltas[t] + gamma * lam * gae * (1 - done[t])
        advantages[t] = gae
    return advantages

# 正确性验证
T_test = 10
r_test = np.random.randn(T_test)
v_test = np.random.randn(T_test + 1)
d_test = np.zeros(T_test)
d_test[-1] = 1

a1 = gae_definition(r_test, v_test, d_test)
a2 = gae_recursive(r_test, v_test, d_test)
print(f"两种方法结果一致: {np.allclose(a1, a2)}")

# 性能测试
trajectory_lengths = [10, 50, 100, 500, 1000, 5000]
times_def = []
times_rec = []

for T in trajectory_lengths:
    r = np.random.randn(T)
    v = np.random.randn(T + 1)
    d = np.zeros(T)
    d[-1] = 1

    t0 = time.perf_counter()
    for _ in range(100):
        gae_definition(r, v, d)
    t1 = time.perf_counter()
    times_def.append(t1 - t0)

    t0 = time.perf_counter()
    for _ in range(100):
        gae_recursive(r, v, d)
    t1 = time.perf_counter()
    times_rec.append(t1 - t0)

# 可视化
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trajectory_lengths, times_def, 'o-', label='定义法 O(T²)', linewidth=2)
ax.plot(trajectory_lengths, times_rec, 's-', label='递推法 O(T)', linewidth=2)
ax.set_xlabel('轨迹长度 T'); ax.set_ylabel('时间 (秒, 100次)')
ax.set_title('GAE 计算复杂度比较')
ax.legend(); ax.grid(True, alpha=0.3)

# 在大 T 时标注
ax.annotate('O(T²) 增长', xy=(trajectory_lengths[-1], times_def[-1]),
            fontsize=9, color='blue')
ax.annotate('O(T) 线性', xy=(trajectory_lengths[-1], times_rec[-1]),
            fontsize=9, color='green')

plt.tight_layout()
plt.savefig('outputs/figures/18_gae_complexity.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/18_gae_complexity.png")

print(f"\n{'T':<8} {'定义法(秒)':<14} {'递推法(秒)':<14} {'加速比':<10}")
for T, td, tr in zip(trajectory_lengths, times_def, times_rec):
    print(f"{T:<8} {td:<14.6f} {tr:<14.6f} {td/tr:<10.2f}x")


## 实际调优指南：如何选择 $\lambda$

### 通用策略

1. **从默认值开始**：$\lambda = 0.95$ 是 PPO/A2C 的标准配置
2. **观察训练曲线**：
   - 如果回报曲线震荡剧烈 → 降低 $\lambda$（更多 bootstrap）
   - 如果回报曲线平滑但偏低 → 提高 $\lambda$（降低偏差）
   - 如果 explained variance 低 → 降低 $\lambda$（减轻价值网络负担）
3. **根据任务调整**：
   - 稀疏奖励 → 低 $\lambda$（0.8-0.9）
   - 密集奖励 → 标准 $\lambda$（0.9-0.95）
   - 长 Horizon → 高 $\lambda$（0.95-0.99）

### $\lambda$ 与其他超参数的联合调优

```
学习率高 → 增大 λ（需要更准确的梯度方向）
              ↓
         λ 调优
              ↑
剪裁 ε 小 → 减小 λ（更新更保守）
```

### 经验法则

| 信号 | $\lambda$ 调整 | 原因 |
|:----:|:--------------:|:------|
| 训练不稳定，回报大幅波动 | 降低 $\lambda$ 0.05-0.1 | 增加 bootstrap，稳定训练 |
| 训练收敛慢，性能不足 | 提高 $\lambda$ 0.02-0.05 | 降低偏差，释放策略潜力 |
| 价值网络误差很大 | 降低 $\lambda$ | 减少对不准确价值函数的依赖 |
| 奖励信号非常清晰 | 提高 $\lambda$ 接近 1 | 更多地依赖实际奖励 |


## 练习

1. **GAE 递推证明**：从原始定义 $A_t = \sum_{l=0}^{\infty} (\gamma\lambda)^l \delta_{t+l}$ 出发，证明递推公式 $A_t = \delta_t + \gamma\lambda A_{t+1}$。

2. **偏差-方差实验**：运行 GAE 在不同 $\lambda$ 下的多次实验，计算每个 $\lambda$ 的偏差和方差，绘制偏差-方差权衡曲线。

3. **$\gamma$ 和 $\lambda$ 的交互**：设计实验同时改变 $\gamma$ 和 $\lambda$，观察它们如何相互作用影响 GAE。

4. **实现 $\lambda$-return**：不依赖优势公式，直接从 $\lambda$-return 的定义计算优势，验证与 GAE 的一致性。

5. **CartPole 的 $\lambda$ 扫描**：在 CartPole 上扫描 $\lambda$ 从 0 到 1 的更多值（如 0.0, 0.25, 0.5, 0.75, 0.95, 1.0），运行更长时间，比较收敛速度和最终性能。
